In [363]:
print(" -------- Task 1 --------- ")
csv_messy_path = r"..\data\messy\messy_market_data.csv"
print(f"loaded: {csv_messy_path}")

 -------- Task 1 --------- 
loaded: ..\data\messy\messy_market_data.csv


In [364]:
import pandas as pd

messy_data_df = pd.read_csv(csv_messy_path)

rows, columns = messy_data_df.shape

print(f"Rows: {rows}")
print(f"Columns: {columns}")

Rows: 9991
Columns: 13


In [365]:
print("First 10 rows of messy data shown:")

first_10_rows_of_messy_data = messy_data_df.head(10)

print(first_10_rows_of_messy_data)

First 10 rows of messy data shown:
      symbol interval                  open_time            open  \
0   BTCUSDT        1h  2026-05-29 23:00:00+00:00  73483.51000000   
1    BTCUSDT       1h  2026-06-07 19:00:00+00:00  62030.15000000   
2   DOGEUSDT       1h  2026-06-23 06:00:00+00:00      0.08172000   
3   LINKUSDT       1h  2026-05-25 18:00:00+00:00      9.60500000   
4    DOTUSDT       1h  2026-05-29 04:00:00+00:00      1.20400000   
5   DOGEUSDT       1h  2026-05-21 08:00:00+00:00      0.10564000   
6   linkusdt       1h  2026-05-31 15:00:00+00:00      9.09900000   
7   LINKUSDT       1h  2026-06-25 09:00:00+00:00      7.50800000   
8   DOGEUSDT       1h  2026-05-22 10:00:00+00:00      0.10575000   
9    BNBUSDT       1h  2026-06-18 10:00:00+00:00    590.67000000   

             high             low           close             volume  \
0  73510.00000000  73320.00000000  73460.78000000       271.23268000   
1  62036.78000000  61184.00000000  61328.00000000      1046.65633000   


In [366]:
messy_data_datatypes = messy_data_df.dtypes

print(messy_data_datatypes)

symbol                    str
interval                  str
open_time                 str
open                      str
high                      str
low                       str
close                     str
volume                    str
close_time                str
quote_volume              str
trade_count               str
taker_buy_base_volume     str
taker_buy_quote_volume    str
dtype: object


In [367]:
print(" -------- Task 2 --------- ")

messy_data_df_missing_counts = messy_data_df.isnull().sum()

sorted_messy_data_missing_counts = messy_data_df_missing_counts.sort_values(ascending=False)

print("All missing values sorted")

print(sorted_messy_data_missing_counts)


 -------- Task 2 --------- 
All missing values sorted
close_time                58
high                      57
quote_volume              51
open_time                 47
trade_count               44
taker_buy_base_volume     43
low                       41
volume                    41
taker_buy_quote_volume    40
open                      38
close                     36
interval                   0
symbol                     0
dtype: int64


In [368]:
print("Most missing data are in the below 3 columns with missing value counts: ")

messy_missing_counts_top_3 = sorted_messy_data_missing_counts.head(3)

print(messy_missing_counts_top_3)

# To get most impacted column: 

messy_missing_counts_top_1 = sorted_messy_data_missing_counts.head(1)

print(f"Most missing data are for most impacted column name: {messy_missing_counts_top_1.index[0]}")

Most missing data are in the below 3 columns with missing value counts: 
close_time      58
high            57
quote_volume    51
dtype: int64
Most missing data are for most impacted column name: close_time


In [369]:
for_cleaning_data_df = messy_data_df.copy()

columns_to_convert_to_numeric = ["open", "close", "high", "low", "trade_count", "volume", "quote_volume", "taker_buy_base_volume", "taker_buy_quote_volume"]

# Get missing ones as mask which keeps marks ones were missing

cleaning_messy_originally_missing_df = for_cleaning_data_df[columns_to_convert_to_numeric].isnull() # to find missing values before coerce changes conversion invalid also as NaN


# Convert and change invalid to NaN also

cleaning_messy_converted_to_numeric = for_cleaning_data_df[columns_to_convert_to_numeric].apply(pd.to_numeric, errors = 'coerce')


# Get inclusive mask with missing and invalid 
Marked_missing_plus_conversion_invalid_as_missing_df = cleaning_messy_converted_to_numeric[columns_to_convert_to_numeric].isnull() # to find as missing - NaN (marked Nan from coerce) also ones from invalid conversion


# Get mask with invalid ones and marks which ones were invalid by subtracting missing mask from total mask

messy_data_only_invalid = Marked_missing_plus_conversion_invalid_as_missing_df & ~ cleaning_messy_originally_missing_df


#

for_cleaning_data_df[columns_to_convert_to_numeric] = cleaning_messy_converted_to_numeric

print("""
Below post conversion datatypes of columns.
The invalid and missing are saved in dataframe masks for reference.
""")

print(for_cleaning_data_df.dtypes)

print(f"""
Attributes converted are: {", ".join(columns_to_convert_to_numeric)}.
""")




Below post conversion datatypes of columns.
The invalid and missing are saved in dataframe masks for reference.

symbol                        str
interval                      str
open_time                     str
open                      float64
high                      float64
low                       float64
close                     float64
volume                    float64
close_time                    str
quote_volume              float64
trade_count               float64
taker_buy_base_volume     float64
taker_buy_quote_volume    float64
dtype: object

Attributes converted are: open, close, high, low, trade_count, volume, quote_volume, taker_buy_base_volume, taker_buy_quote_volume.



In [370]:
# Now have to count rows which have at least one NaN or true for isnull() as opposed to cells.
# Marked_missing_plus_conversion_invalid_as_missing_df This True False Dataframe has True for missing or conversion_invalid. We group them all as invalid/missing

# print(Marked_missing_plus_conversion_invalid_as_missing_df.sum()) # We sum the invalid/missing -- But this would be by column we want by row

check_marked_missing_plus_invalid_in_row = Marked_missing_plus_conversion_invalid_as_missing_df.any(axis=1) # We check if invalid/missing (True) on the row - req is by row and not by column hence we focus on dataframe axis

missing_invalid_row_count = check_marked_missing_plus_invalid_in_row.sum()

print(f"Invalid/Missing numeric rows after conversion: {missing_invalid_row_count}.")

Invalid/Missing numeric rows after conversion: 770.


In [371]:
columns_convert_to_timestamp = ["open_time", "close_time"]

print("""
Task 4: Converting time attributes to timestamp. 
Then marking invalid to conversion values for capturing.
""")

cleaning_messy_converted_to_timestamp_df = for_cleaning_data_df[columns_convert_to_timestamp].apply(pd.to_datetime, errors = 'coerce')

print(f"Converted to timestamp columns: {", ".join(columns_convert_to_timestamp)}")

# To count the invalid values which are Nan after the coerse I do isnull()

cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True = cleaning_messy_converted_to_timestamp_df.isnull() # mark invalid ones from conversion to timestamp as true

print("After conversion, below are invalid to timestamp conersion counts:")

open_time_invalid_counts, close_time_invalid_counts = cleaning_messy_converted_to_timestamp_mark_invalid_ones_as_True.sum()

print(f"Invalid open_time counts: {open_time_invalid_counts}.")
print(f"Invalid close_time counts: {close_time_invalid_counts}.\n")

for_cleaning_data_df[columns_convert_to_timestamp] = cleaning_messy_converted_to_timestamp_df # Put the applied with conversion df into the cleaned df.

print("Validating after new conversion cleaned df datatypes:\n")
print(for_cleaning_data_df.dtypes)



Task 4: Converting time attributes to timestamp. 
Then marking invalid to conversion values for capturing.

Converted to timestamp columns: open_time, close_time
After conversion, below are invalid to timestamp conersion counts:
Invalid open_time counts: 199.
Invalid close_time counts: 205.

Validating after new conversion cleaned df datatypes:

symbol                                    str
interval                                  str
open_time                 datetime64[us, UTC]
open                                  float64
high                                  float64
low                                   float64
close                                 float64
volume                                float64
close_time                datetime64[us, UTC]
quote_volume                          float64
trade_count                           float64
taker_buy_base_volume                 float64
taker_buy_quote_volume                float64
dtype: object


In [372]:
# Now cleaning and standardising the format of values of symbol attribute:

print("Now cleaning and standardising the format of values of symbol attribute.\n")

symbol_columns_for_cleaning_and_standardization = ["symbol"]

# Printing symbol values before cleaning:

print("Below are symbol value counts before cleaning:")


print(for_cleaning_data_df["symbol"].value_counts())


unique_symbol_values_before_cleaning = for_cleaning_data_df["symbol"].sort_values().unique() # Symbol values before cleaning.

print(f"\nSymbol values before cleaning: {", ".join(unique_symbol_values_before_cleaning)}")

# Below are the cleaning processing steps for symbol:

cleaning_and_standardizing_symbol_values_df = for_cleaning_data_df["symbol"].str.strip().str.upper().str.replace("/","")

for_cleaning_data_df["symbol"] = cleaning_and_standardizing_symbol_values_df

# Printing

print("\nBelow are symbol value counts after cleaning:")

print(f"{for_cleaning_data_df["symbol"].value_counts()}\n")

unique_symbol_values_after_cleaning = for_cleaning_data_df["symbol"].sort_values().unique()  # Symbol values after cleaning.

print(f"Symbol values after cleaning: {', '.join(unique_symbol_values_after_cleaning)}\n")

print(f"Unique symbols count after cleaning: {len(unique_symbol_values_after_cleaning)}")

Now cleaning and standardising the format of values of symbol attribute.

Below are symbol value counts before cleaning:
symbol
AVAXUSDT      978
XRPUSDT       970
DOTUSDT       965
ETHUSDT       964
ADAUSDT       963
DOGEUSDT      959
BNBUSDT       955
LINKUSDT      952
BTCUSDT       950
SOLUSDT       936
BTC/USDT       30
SOL/USDT       21
ADA/USDT       19
linkusdt       18
ETH/USDT       18
DOGE/USDT      17
 AVAXUSDT      17
 ADAUSDT       15
 ETHUSDT       15
 SOLUSDT       14
avaxusdt       14
ethusdt        14
LINK/USDT      14
adausdt        14
DOT/USDT       13
 DOGEUSDT      12
XRP/USDT       12
BNB/USDT       11
AVAX/USDT      11
dotusdt        10
btcusdt        10
dogeusdt       10
 LINKUSDT      10
 BNBUSDT       10
solusdt         9
 XRPUSDT        9
 DOTUSDT        9
 BTCUSDT        8
xrpusdt         8
bnbusdt         7
Name: count, dtype: int64

Symbol values before cleaning:  ADAUSDT ,  AVAXUSDT ,  BNBUSDT ,  BTCUSDT ,  DOGEUSDT ,  DOTUSDT ,  ETHUSDT ,  LINKUSDT ,  SO

In [373]:
# Task 5 for Counting and removing duplicate rows.

print ("""
---- Task 5 ---
Counting and removing duplicates.""")

duplicates_count = for_cleaning_data_df.duplicated().sum()

print(f"Duplicate rows found: {duplicates_count}.")

#Row counts before duplicates removed.

messy_rows_counts_before_duplicates_removal = len(for_cleaning_data_df)

rows_after_duplicates_removal = for_cleaning_data_df.drop_duplicates()

rows_counts_after_duplicates_removal = len(rows_after_duplicates_removal)

for_cleaning_data_df = rows_after_duplicates_removal

print(f"Rows before removing duplicates: {messy_rows_counts_before_duplicates_removal}.")

print(f"Rows after removing duplicates: {rows_counts_after_duplicates_removal}.")

# print(len(for_cleaning_data_df)) # For checking the values of for_cleaning_data_df after duplicate rows removal



---- Task 5 ---
Counting and removing duplicates.
Duplicate rows found: 213.
Rows before removing duplicates: 9991.
Rows after removing duplicates: 9778.
